# 01 · Preprocesamiento: de datos crudos a descriptores moleculares

Este notebook parte de cuatro archivos crudos —AqSolDB, Delaney/ESOL y los
dos conjuntos del Second Solubility Challenge (SC-2: `tight` y `loose`)— y
produce los conjuntos limpios, con descriptores 2D de RDKit, que usa el resto
del proyecto.

**Referencias de los datasets:**

> Sorkun, M.C., Khetan, A. & Er, S. (2019). *AqSolDB, a curated reference set
> of aqueous solubility and 2D descriptors for a diverse set of compounds.*
> Scientific Data, 6(1), 143. https://doi.org/10.1038/s41597-019-0151-1

> Delaney, J.S. (2004). *ESOL: Estimating aqueous solubility directly from
> molecular structure.* J. Chem. Inf. Comput. Sci., 44(3), 1000–1005.

> Conn, J.G.M., Carter, J.W., Brémond, E., et al. (2023). *Blinded Predictions
> and Post-hoc Analysis of the Second Solubility Challenge Data: Exploring
> Training Set Size and Solubility Range.* J. Chem. Inf. Model.
> https://doi.org/10.1021/acs.jcim.2c01189

**Cálculo de descriptores moleculares:** los descriptores 2D se obtienen mediante RDKit, una biblioteca de química computacional y quimioinformática de código abierto (Landrum, s. f.).

## Por qué se filtra el solapamiento entre datasets

AqSolDB es, por diseño, una compilación de nueve fuentes de solubilidad
públicas — y una de esas nueve fuentes es, precisamente, Delaney/ESOL. Además,
el conjunto de entrenamiento D5697 construido por Conn et al. (2023) para el
análisis post-hoc del Second Solubility Challenge se derivó también de
AqSolDB, tras eliminar las moléculas de test. Esto implica que AqSolDB puede
solaparse no solo con Delaney, sino también con los conjuntos `tight` y
`loose` del SC-2 como se demuestra más adelante.

Si no se corrige esto, entrenar con AqSolDB y "validar externamente" con
cualquiera de los tres conjuntos significa evaluar, en parte, moléculas que el
modelo ya vio en entrenamiento, por tanto no estariamos midiendo una capacidad de 
generalizacion , sino de memorizacion y ocurriria una fuga de datos (*data leakage*) que
invalida la comparación con modelos internacionales.

Este notebook identifica esas moléculas por su **InChIKey** (un identificador
químico único, más confiable que comparar el texto del SMILES) y las retira
del conjunto de entrenamiento. Los tres conjuntos de validación externa
—Delaney/ESOL, SC-2 `tight` y SC-2 `loose`— se mantienen **completos e
intactos**, listos para una evaluación ciega y genuina contra los benchmarks
publicados por Conn et al. (2023).

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from rdkit import Chem
from rdkit.Chem import Descriptors, inchi
from tqdm import tqdm

RANDOM_STATE = 1
np.random.seed(RANDOM_STATE)

print("Configuración lista")

Configuración lista


## Paso 1 · Cargar los datos crudos

In [2]:
df_aqsoldb_crudo = pd.read_csv('../data/raw/data_aqsoldb.csv')
df_delaney_crudo = pd.read_csv('../data/raw/data_delaney.csv')
df_tight_crudo = pd.read_csv('../data/raw/Tight_set.csv')
df_loose_crudo = pd.read_csv('../data/raw/Loose_set.csv')

print(f"AqSolDB crudo:  {len(df_aqsoldb_crudo):,} moléculas, {df_aqsoldb_crudo.shape[1]} columnas")
print(f"Delaney crudo:  {len(df_delaney_crudo):,} moléculas, {df_delaney_crudo.shape[1]} columnas")
print(f"Tight de SC-2 crudo:  {len(df_tight_crudo):,} moléculas, {df_tight_crudo.shape[1]} columnas")
print(f"Loose de SC-2 crudo:  {len(df_loose_crudo):,} moléculas, {df_loose_crudo.shape[1]} columnas")

# AqSolDB, Tight_SC-2 y Loose_SC-2 ya trae descriptores 2D precalculados, se descartan aqui
# porque en el paso 3 se recalculan todos los descriptores de RDKit de forma
# uniforme, para no mezclar dos fuentes de calculo distintas.
columnas_descriptores_previos = [
    'MolWt', 'MolLogP', 'MolMR', 'HeavyAtomCount', 'NumHAcceptors', 'NumHDonors',
    'NumHeteroatoms', 'NumRotatableBonds', 'NumValenceElectrons', 'NumAromaticRings',
    'NumSaturatedRings', 'NumAliphaticRings', 'RingCount', 'TPSA', 'LabuteASA',
    'BalabanJ', 'BertzCT'
]
df_aqsoldb_crudo = df_aqsoldb_crudo.drop(columns=columnas_descriptores_previos)

columnas_descriptores_delaney = [
    'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors',
    'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area'
]
df_delaney_crudo = df_delaney_crudo.drop(columns=columnas_descriptores_delaney)

def limpiar_sets_SC(df, colum_id, colum_name, colum_smile, colum_solubilidad):
    df = df.iloc[1:].reset_index(drop=True) #Eliminar el encabezado duplicado
    columnas = [colum_id, colum_name, colum_smile, colum_solubilidad]
    df_limpio = df[columnas].rename(columns = {colum_id:'ID', colum_name:'Name', colum_smile:'smiles', 
                                              colum_solubilidad:'Solubility'}).copy()
    return df_limpio
    
df_tight_crudo = limpiar_sets_SC(df_tight_crudo, 'ID' , 'Name', 'SMILES', 'Solubility' )
df_loose_crudo = limpiar_sets_SC(df_loose_crudo, 'ID' , 'Name', 'SMILES', 'Solubility' )

AqSolDB crudo:  9,982 moléculas, 26 columnas
Delaney crudo:  1,128 moléculas, 10 columnas
Tight de SC-2 crudo:  101 moléculas, 1331 columnas
Loose de SC-2 crudo:  33 moléculas, 1331 columnas


## Paso 2 · Filtrar el solapamiento SC-2 ↔ AqSolDB ↔ Delaney

AqSolDB ya incluye su propio InChIKey (columna `InChIKey`). Para Delaney y los demas solo
hace falta calcularlo a partir del SMILES.

In [3]:
def smiles_a_inchikey(smiles):
    """Convierte un SMILES a InChIKey; devuelve None si el SMILES es invalido."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return inchi.MolToInchiKey(mol)
    except Exception:
        return None

#Calcular InChIKeys de los tres conjuntos de validación externa
tqdm.pandas()

print("Calculando InChIKeys de los conjuntos de validación externa...")

inchikeys_delaney = set(
    df_delaney_crudo['smiles'].progress_apply(smiles_a_inchikey).dropna()
)
inchikeys_tight = set(
    df_tight_crudo['smiles'].progress_apply(smiles_a_inchikey).dropna()
)
inchikeys_loose = set(
    df_loose_crudo['smiles'].progress_apply(smiles_a_inchikey).dropna()
)

# Unión de los tres: cualquier molécula que aparezca en cualquiera
# de los tres conjuntos de validación externa debe salir del entrenamiento.
inchikeys_validacion_externa = inchikeys_delaney | inchikeys_tight | inchikeys_loose

#Filtrado único contra la unión
n_antes = len(df_aqsoldb_crudo)
solapadas = df_aqsoldb_crudo['InChIKey'].isin(inchikeys_validacion_externa)

print("\n" + "="*60)
print("REPORTE DE SOLAPAMIENTO AqSolDB <-> VALIDACIÓN EXTERNA")
print("="*60)
print(f"AqSolDB antes de filtrar:                    {n_antes:,}")
print(f"  · solapadas con Delaney:                   "
      f"{df_aqsoldb_crudo['InChIKey'].isin(inchikeys_delaney).sum():,}")
print(f"  · solapadas con Tight_SC-2:                "
      f"{df_aqsoldb_crudo['InChIKey'].isin(inchikeys_tight).sum():,}")
print(f"  · solapadas con Loose_SC-2:                "
      f"{df_aqsoldb_crudo['InChIKey'].isin(inchikeys_loose).sum():,}")
print(f"  · solapadas con la unión en total (a eliminar):     {solapadas.sum():,}")
print(f"AqSolDB tras el filtro:                      {(~solapadas).sum():,}")
print("-"*60)
print(f"Delaney intacto:                             {len(df_delaney_crudo):,}")
print(f"Tight_SC-2 intacto:                          {len(df_tight_crudo):,}")
print(f"Loose_SC-2 intacto:                          {len(df_loose_crudo):,}")

df_aqsoldb_crudo = df_aqsoldb_crudo[~solapadas].reset_index(drop=True)

Calculando InChIKeys de los conjuntos de validación externa...


100%|█████████████████████████████████████████████████████████████████████████████████| 32/32 [00:00<00:00, 662.77it/s]



REPORTE DE SOLAPAMIENTO AqSolDB <-> VALIDACIÓN EXTERNA
AqSolDB antes de filtrar:                    9,982
  · solapadas con Delaney:                   1,117
  · solapadas con Tight_SC-2:                74
  · solapadas con Loose_SC-2:                15
  · solapadas con la unión en total (a eliminar):     1,190
AqSolDB tras el filtro:                      8,792
------------------------------------------------------------
Delaney intacto:                             1,128
Tight_SC-2 intacto:                          100
Loose_SC-2 intacto:                          32


## Paso 3 · Calcular descriptores moleculares 2D con RDKit

Se calculan todos los descriptores 2D disponibles en RDKit (`Descriptors.CalcMolDescriptors`,
más de 100 propiedades topológicas y fisicoquímicas) para cada molécula, a
partir únicamente de su SMILES.

In [4]:
def calcular_descriptores_rdkit(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Descriptors.CalcMolDescriptors(mol)


print("Calculando descriptores para AqSolDB...")
tqdm.pandas(desc='Descriptores AqSolDB')
descriptores_aqsoldb = df_aqsoldb_crudo['SMILES'].progress_apply(calcular_descriptores_rdkit)
df_aqsoldb = pd.concat([df_aqsoldb_crudo, pd.DataFrame(descriptores_aqsoldb.tolist())], axis=1)
df_aqsoldb.dropna(inplace=True)
print(f"AqSolDB final: {df_aqsoldb.shape[0]:,} moléculas, {df_aqsoldb.shape[1]} columnas")

print("\nCalculando descriptores para Delaney...")
tqdm.pandas(desc='Descriptores Delaney')
descriptores_delaney = df_delaney_crudo['smiles'].progress_apply(calcular_descriptores_rdkit)
df_delaney = pd.concat([df_delaney_crudo, pd.DataFrame(descriptores_delaney.tolist())], axis=1)
df_delaney.dropna(inplace=True)
print(f"Delaney final: {df_delaney.shape[0]:,} moléculas, {df_delaney.shape[1]} columnas")

print("\nCalculando descriptores para Tight_SC-2...")
tqdm.pandas(desc='Descriptores Tight_SC-2')
descriptores_tight = df_tight_crudo['smiles'].progress_apply(calcular_descriptores_rdkit)
df_tight = pd.concat([df_tight_crudo, pd.DataFrame(descriptores_tight.tolist())], axis=1)
df_tight.dropna(inplace=True)
print(f"Tight_SC-2 final: {df_tight.shape[0]:,} moléculas, {df_tight.shape[1]} columnas")

print("\nCalculando descriptores para Loose_SC-2...")
tqdm.pandas(desc='Descriptores Loose_SC-2')
descriptores_loose = df_loose_crudo['smiles'].progress_apply(calcular_descriptores_rdkit)
df_loose = pd.concat([df_loose_crudo, pd.DataFrame(descriptores_loose.tolist())], axis=1)
df_loose.dropna(inplace=True)
print(f"Loose_SC-2 final: {df_loose.shape[0]:,} moléculas, {df_loose.shape[1]} columnas")

Calculando descriptores para AqSolDB...


Descriptores AqSolDB: 100%|████████████████████████████████████████████████████████| 8792/8792 [02:59<00:00, 49.05it/s]


AqSolDB final: 7,908 moléculas, 134 columnas

Calculando descriptores para Delaney...


Descriptores Delaney: 100%|████████████████████████████████████████████████████████| 1128/1128 [00:15<00:00, 72.18it/s]


Delaney final: 1,128 moléculas, 129 columnas

Calculando descriptores para Tight_SC-2...


Descriptores Tight_SC-2: 100%|███████████████████████████████████████████████████████| 100/100 [00:02<00:00, 40.02it/s]


Tight_SC-2 final: 100 moléculas, 129 columnas

Calculando descriptores para Loose_SC-2...


Descriptores Loose_SC-2: 100%|█████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 23.43it/s]

Loose_SC-2 final: 32 moléculas, 129 columnas


## Paso 4 · Separar una validación interna y guardar

Del subconjunto de mayor calidad de AqSolDB (`SD == 0`, sin desviación
estándar reportada entre mediciones repetidas) se aparta un 20% como
**validación interna**: mismo origen que el entrenamiento, útil para detectar
sobreajuste temprano, pero distinto de la validación externa que se quiere realizar.

In [5]:
alta_calidad = df_aqsoldb['SD'] == 0
df_alta_calidad = df_aqsoldb[alta_calidad].copy()
df_resto = df_aqsoldb[~alta_calidad].copy()

df_validacion_interna, df_alta_calidad_entrenamiento = train_test_split(
    df_alta_calidad, test_size=0.8, random_state=RANDOM_STATE
)

df_entrenamiento = pd.concat([df_resto, df_alta_calidad_entrenamiento], ignore_index=True)

print(f"Entrenamiento:        {len(df_entrenamiento):,} moléculas")
print(f"Validación interna:   {len(df_validacion_interna):,} moléculas")
print(f"Validación externa:   {len(df_delaney):,} moléculas (Delaney, sin solapamiento)")
print(f"Validación externa:   {len(df_tight):,} moléculas (Tight_SC-2, sin solapamiento)")
print(f"Validación externa:   {len(df_loose):,} moléculas (Loose_SC-2, sin solapamiento)")


os.makedirs('../data/processed', exist_ok=True)
df_entrenamiento.to_csv('../data/processed/df_aqsoldb_entrenamiento.csv', index=False)
df_validacion_interna.to_csv('../data/processed/df_aqsoldb_validacion_externa.csv', index=False)
df_delaney.to_csv('../data/processed/df_delaney_complete.csv', index=False)
df_tight.to_csv('../data/processed/df_tight_complete.csv', index=False)
df_loose.to_csv('../data/processed/df_loose_complete.csv', index=False)

print("\nArchivos guardados en ../data/processed/")

Entrenamiento:        6,592 moléculas
Validación interna:   1,316 moléculas
Validación externa:   1,128 moléculas (Delaney, sin solapamiento)
Validación externa:   100 moléculas (Tight_SC-2, sin solapamiento)
Validación externa:   32 moléculas (Loose_SC-2, sin solapamiento)

Archivos guardados en ../data/processed/
